In [3]:
cd /data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/

/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY


/data/home/wangzz_group/zhaipengyuan/.conda/envs/BEPH/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### 提取所有数据集特征

In [5]:
import os
import time
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmselfsup.apis import init_model

# ================= 1. 全局配置区域 =================
# 数据根目录 (请根据上一部的设置确认)
ROOT_DIR = './kz_data' 

# 输入目录
PATCHES_DIR = os.path.join(ROOT_DIR, 'Segmentation', 'patches') # .h5 坐标文件
IMAGES_DIR = os.path.join(ROOT_DIR, 'Raw_Data', 'images')       # .png 原始图片
CSV_PATH = 'process_list.csv'                                   # 样本列表

# 输出目录
OUTPUT_DIR = os.path.join(ROOT_DIR, 'BEPH_Features')
os.makedirs(os.path.join(OUTPUT_DIR, 'h5_files'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'pt_files'), exist_ok=True)

# 模型配置 (保持你提供的路径不变)
MODEL_CONFIG = '/data/home/wangzz_group/zhaipengyuan/BEPH-main/mmselfsup/configs/tsne/beitv2_base.py'
MODEL_CHECKPOINT = '/data/home/wangzz_group/zhaipengyuan/BEPH-main/checkpoints/BEPH_backbone.pth'

# 参数设置
TARGET_PATCH_SIZE = 224  # 模型输入大小
BATCH_SIZE = 128         # 显存允许的话建议调大 (48 -> 128 或 256) 加速
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# =======================================================

# --- 辅助类：模拟 OpenSlide ---
class PILSlide:
    def __init__(self, path):
        Image.MAX_IMAGE_PIXELS = None
        self.img = Image.open(path).convert('RGB')
        self.level_dimensions = [self.img.size] 
        
    def read_region(self, location, level, size):
        x, y = location
        w, h = size
        return self.img.crop((x, y, x + w, y + h))

# --- Dataset 定义 ---
def eval_transforms(mean, std):
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

class Whole_Slide_Bag_FP(Dataset):
    def __init__(self, file_path, wsi, target_patch_size=224):
        self.wsi = wsi
        self.file_path = file_path
        self.target_patch_size = (target_patch_size, target_patch_size)
        self.roi_transforms = transforms.Compose([transforms.ToTensor(),
                                                  transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])

        with h5py.File(self.file_path, "r") as f:
            dset = f['coords']
            self.patch_size = f['coords'].attrs['patch_size']
            self.patch_level = f['coords'].attrs['patch_level']
            self.length = len(dset)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.file_path, 'r') as hdf5_file:
            coord = hdf5_file['coords'][idx]
        
        # 1. 从原图切取 (48x48)
        img = self.wsi.read_region(coord, self.patch_level, (self.patch_size, self.patch_size)).convert('RGB')
        
        # 2. 放大到模型需求 (224x224)
        if self.target_patch_size is not None:
            img = img.resize(self.target_patch_size, Image.Resampling.LANCZOS)
            
        img = self.roi_transforms(img)
        return img, coord

def collate_features(batch):
    img = torch.stack([item[0] for item in batch], dim=0)
    coords = np.vstack([item[1] for item in batch])
    return [img, coords]

# ================= 核心处理逻辑 (修正版：增加保存 .pt) =================
def process_one_slide(model, slide_id, h5_path, slide_path):
    output_h5_name = os.path.join(OUTPUT_DIR, 'h5_files', slide_id + '.h5')
    output_pt_name = os.path.join(OUTPUT_DIR, 'pt_files', slide_id + '.pt') # <--- 新增
    
    # 跳过检查 (如果两个都存在才跳过)
    if os.path.exists(output_h5_name) and os.path.exists(output_pt_name):
        print(f"⏩ [跳过] {slide_id} 已存在。")
        return

    try:
        # 1. 准备数据
        wsi = PILSlide(slide_path)
        dataset = Whole_Slide_Bag_FP(file_path=h5_path, wsi=wsi, target_patch_size=TARGET_PATCH_SIZE)
        
        if len(dataset) == 0:
            print(f"⚠️ [跳过] {slide_id} 没有任何 patch。")
            return

        loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, collate_fn=collate_features)
        
        # 2. 提取特征
        features_list = []
        coords_list = []
        
        for batch, coords in tqdm(loader, desc=f"Ext: {slide_id}", leave=False):
            with torch.no_grad():
                batch = batch.to(DEVICE)
                
                # mmselfsup 提取特征
                if hasattr(model, 'module'):
                    feat = model.module.extract_feat(batch, stage='backbone')[0]
                else:
                    feat = model.extract_feat(batch, stage='backbone')[0]
                
                if len(feat.shape) == 3: 
                    feat = feat[:, 0, :] 
                
                features_list.append(feat.cpu().numpy().astype(np.float32)) 
                coords_list.append(coords)

        # 3. 整合
        all_features = np.concatenate(features_list, axis=0)
        all_coords = np.concatenate(coords_list, axis=0)
        
        # 4. 保存 .h5 (特征 + 坐标)
        with h5py.File(output_h5_name, 'w') as f:
            f.create_dataset('features', data=all_features)
            f.create_dataset('coords', data=all_coords)
            
        # 5. 保存 .pt (纯特征，用于快速训练) <--- 新增部分
        torch.save(torch.from_numpy(all_features), output_pt_name)
            
        print(f"✅ 完成: {slide_id} -> Shape: {all_features.shape}")

    except Exception as e:
        print(f"❌ 错误 {slide_id}: {e}")
# ================= 主程序入口 =================
if __name__ == "__main__":
    # 1. 加载模型 (只加载一次！)
    print("正在加载 BEPH 模型...")
    try:
        cfg = Config.fromfile(MODEL_CONFIG)
        init_default_scope(cfg.get('default_scope', 'mmselfsup'))
        model = init_model(cfg, MODEL_CHECKPOINT, device=DEVICE)
        model.eval()
        print("Model loaded successfully.")
    except Exception as e:
        print(f"❌ 模型加载失败: {e}")
        exit()

    # 2. 读取列表
    if not os.path.exists(CSV_PATH):
        print(f"CSV文件未找到: {CSV_PATH}")
        exit()
        
    df = pd.read_csv(CSV_PATH)
    ids = df.iloc[:, 0].tolist() if 'slide_id' not in df.columns else df['slide_id'].tolist()
    
    print(f"🚀 开始批量提取特征，共 {len(ids)} 个样本...")
    
    # 3. 循环处理
    for slide_id in tqdm(ids, desc="Total Progress"):
        # 构造路径
        h5_file = os.path.join(PATCHES_DIR, f"{slide_id}.h5")
        img_file = os.path.join(IMAGES_DIR, f"{slide_id}.png")
        
        if not os.path.exists(h5_file):
            print(f"❌ 缺失 Patch 坐标文件: {h5_file}")
            continue
        if not os.path.exists(img_file):
            print(f"❌ 缺失图片文件: {img_file}")
            continue
            
        process_one_slide(model, str(slide_id), h5_file, img_file)
        
    print("\n✨ 全部任务结束！")
    print(f"特征已保存在: {OUTPUT_DIR}")

正在加载 BEPH 模型...


/data/home/wangzz_group/zhaipengyuan/.conda/envs/BEPH/lib/python3.9/importlib/__init__.py:169: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  _bootstrap._exec(spec, module)


Loads checkpoint by local backend from path: /data/home/wangzz_group/zhaipengyuan/BEPH-main/checkpoints/BEPH_backbone.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: backbone.mask_token, backbone.rel_pos_bias.relative_position_bias_table, backbone.rel_pos_bias.relative_position_index

missing keys in source state_dict: backbone.layers.0.attn.relative_position_bias_table, backbone.layers.0.attn.relative_position_index, backbone.layers.1.attn.relative_position_bias_table, backbone.layers.1.attn.relative_position_index, backbone.layers.2.attn.relative_position_bias_table, backbone.layers.2.attn.relative_position_index, backbone.layers.3.attn.relative_position_bias_table, backbone.layers.3.attn.relative_position_index, backbone.layers.4.attn.relative_position_bias_table, backbone.layers.4.attn.relative_position_index, backbone.layers.5.attn.relative_position_bias_table, backbone.layers.5.attn.relative_position_index, backbone.layers.6.attn.rel

Total Progress:   2%|▏         | 1/65 [00:09<10:17,  9.64s/it]                

✅ 完成: ProstateAcinarCancer_10x -> Shape: (4992, 768)


...

✅ 完成: GSM5621965 -> Shape: (407, 768)



Total Progress:  98%|█████████▊| 64/65 [05:36<00:03,  3.97s/it]           

✅ 完成: ColorectalCancer_10x -> Shape: (4672, 768)



Total Progress: 100%|██████████| 65/65 [05:44<00:00,  5.29s/it]

✅ 完成: 151673 -> Shape: (3639, 768)

✨ 全部任务结束！
特征已保存在: ./kz_data/BEPH_Features
